In [1]:
%%capture
!pip install unsloth modelscope
!pip install --force-reinstall "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "datasets<4.4.0"

In [1]:
import os
import torch
from unsloth import FastLanguageModel
from huggingface_hub import login
from google.colab import userdata

# 1. Set environment and authenticate
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
login(token=userdata.get('HF_TOKEN'))

# 2. Load pre-quantized model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

print("✅ Model successfully loaded via ModelScope Mirror!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-instruct-bnb-4bit as a legacy tokenizer.


✅ Model successfully loaded via ModelScope Mirror!


In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ LoRA adapters successfully injected!")

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA adapters successfully injected!


In [3]:
import json
from datasets import Dataset

file_path = "/content/chain_of_thought_dataset.jsonl"
final_data = []

# Parse JSONL robustly
with open(file_path, 'r', encoding='utf-8') as f:
    for line_number, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            final_data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"⚠️ Skipping line {line_number} due to error: {e}")

dataset = Dataset.from_list(final_data)
print(f"✅ Success! Loaded {len(dataset)} examples.")

# Apply Llama 3 Chat Template
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    thinkings    = examples["thinking"]
    responses    = examples["response"]
    texts = []

    for i, t, r in zip(instructions, thinkings, responses):
        text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{i}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{t}\n{r}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("✅ Formatting complete!")

⚠️ Skipping line 60 due to error: Extra data: line 1 column 1166 (char 1165)
✅ Success! Loaded 475 examples.


Map:   0%|          | 0/475 [00:00<?, ? examples/s]

✅ Formatting complete!


In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 3,
        warmup_ratio = 0.1,
        logging_steps = 10,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Starting training loop...")
trainer_stats = trainer.train()
print("✅ Training complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/475 [00:00<?, ? examples/s]

🚀 Starting training loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 475 | Num Epochs = 3 | Total steps = 180
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.760223
20,1.023184
30,0.711010
40,0.586098
50,0.554469
60,0.482652
70,0.289223
80,0.291928
90,0.358822
100,0.429170


✅ Training complete!


In [5]:
FastLanguageModel.for_inference(model)

test_question = "A snail is at the bottom of a 20-foot well. Every day, it climbs up 5 feet. Every night, while asleep, it slides back down 4 feet. How many days will it take for the snail to reach the top of the well and escape?"

prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{test_question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

print("\n--- MODEL OUTPUT ---\n")
print(decoded_output.split("assistant\n\n")[-1])

Both `max_new_tokens` (=1024) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



--- MODEL OUTPUT ---

<thinking>This is a classic math word problem. I need to track the snail's progress each day.

Day 1: The snail climbs 5 feet, so it's at 5 feet. Then it slips back 4 feet, so it's at 1 foot.
Day 2: The snail climbs 5 feet, so it's at 6 feet. Then it slips back 4 feet, so it's at 2 feet.
Day 3: The snail climbs 5 feet, so it's at 7 feet. Then it slips back 4 feet, so it's at 3 feet.
Day 4: The snail climbs 5 feet, so it's at 8 feet. Then it slips back 4 feet, so it's at 4 feet.
Day 5: The snail climbs 5 feet, so it's at 9 feet. Then it slips back 4 feet, so it's at 5 feet.
Day 6: The snail climbs 5 feet, so it's at 10 feet. Then it slips back 4 feet, so it's at 6 feet.
Day 7: The snail climbs 5 feet, so it's at 11 feet. Then it slips back 4 feet, so it's at 7 feet.
Day 8: The snail climbs 5 feet, so it's at 12 feet. Then it slips back 4 feet, so it's at 8 feet.
Day 9: The snail climbs 5 feet, so it's at 13 feet. Then it slips back 4 feet, so it's at 9 feet.
Day 1

In [6]:
from unsloth import FastLanguageModel
import torch

print("⏳ Loading Original Base Model...")
# 1. Load the ORIGINAL Base Model (No LoRA adapters attached)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model) # Optimize for fast chat

# 2. Set up your test question (No <thinking> tags added!)
test_question = "A snail is at the bottom of a 20-foot well. Every day, it climbs up 5 feet. Every night, while asleep, it slides back down 4 feet. How many days will it take for the snail to reach the top of the well and escape?"

# 3. Format using standard Llama 3 chat template
prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{test_question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 4. Generate the "Before" answer
print("🧠 Generating answer...")
outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

print("\n--- ORIGINAL BASE MODEL OUTPUT ---\n")
print(decoded_output.split("assistant\n\n")[-1])

⏳ Loading Original Base Model...
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=1024) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 Generating answer...

--- ORIGINAL BASE MODEL OUTPUT ---

A classic problem!

Let's break it down:

Day 1: The snail climbs 5 feet, so it's at 5 feet above the bottom of the well.
Night 1: It slides back down 4 feet, so it's back at the bottom of the well.

Day 2: The snail climbs 5 feet, so it's at 10 feet above the bottom of the well.
Night 2: It slides back down 4 feet, so it's at 6 feet above the bottom of the well.

Day 3: The snail climbs 5 feet, so it's at 11 feet above the bottom of the well.
Night 3: It slides back down 4 feet, so it's at 7 feet above the bottom of the well.

Notice a pattern? The snail is making progress upwards, but at a slower rate due to the nightly slide back down.

Let's calculate the progress:

Day 1: 5 feet
Day 2: 5 feet (total: 10 feet)
Day 3: 5 feet (total: 15 feet)
Day 4: 5 feet (total: 20 feet)

The snail reaches the top of the well on Day 4. It takes 4 days for the snail to reach the top of the well and escape.

So, the answer is: 4 days.
